# 📄 Marker PDF Parser
### Google Colab Notebook (T4 GPU)

Converts PDF files to **markdown, JSON, HTML, or chunks** using [Marker](https://github.com/VikParuchuri/marker).

- Handles text, tables, equations, images, code blocks
- Works in all languages
- Optionally boosts accuracy with an LLM (Gemini, Claude, Ollama, OpenAI)
- ~3.17 GB VRAM on T4

---
**Steps:**
1. Check GPU
2. Install dependencies
3. Mount Google Drive
4. Configure & run

## 1. 🖥️ Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Sat Mar 21 19:43:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             32W /   70W |    6941MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. 📦 Install Dependencies

In [ ]:
# marker-pdf[full] includes support for PPTX, DOCX, XLSX, HTML, EPUB in addition to PDF
!pip install -q marker-pdf[full]
print('✅ Marker installed.')

✅ Marker installed.


## 3. 📁 Mount Google Drive (optional)

In [ ]:
USE_GOOGLE_DRIVE = True  # Set False to upload files directly instead

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive mounted at /content/drive')
else:
    print('ℹ️  Drive not mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted at /content/drive


In [ ]:
# Run this cell only if NOT using Google Drive
UPLOAD_FROM_LOCAL = False

if UPLOAD_FROM_LOCAL:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        print(f'✅ Uploaded: {fname}')

## 4. ⚙️ Configuration

In [ ]:
import os

# ── Input file ────────────────────────────────────────────────────────────────
# Supports: PDF, image, PPTX, DOCX, XLSX, HTML, EPUB
# Google Drive: '/content/drive/MyDrive/your_file.pdf'
# Local upload: '/content/your_file.pdf'
INPUT_FILE  = '/content/drive/MyDrive/tyt-biyoloji.pdf'   # ← CHANGE THIS

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR  = '/content/drive/MyDrive/marker_output'

# ── Output format ─────────────────────────────────────────────────────────────
# 'markdown' : markdown text + images (default, best for reading)
# 'json'     : structured tree with bounding boxes
# 'html'     : HTML with equations and tables
# 'chunks'   : flat list of blocks, ideal for RAG pipelines
OUTPUT_FORMAT = 'markdown'

# ── Page range ────────────────────────────────────────────────────────────────
# None  → all pages
# '0,5-10,20' → pages 0, 5 through 10, and 20  (0-indexed)
PAGE_RANGE  = (9, 172)

# ── LLM boost ─────────────────────────────────────────────────────────────────
# Set True to use an LLM to improve accuracy (tables, math, forms).
# Requires configuring an LLM service below.
USE_LLM     = False

# ── OCR options ───────────────────────────────────────────────────────────────
# True  → force OCR on every page (use if you see garbled text)
# False → use embedded text where available (faster)
FORCE_OCR   = False

# ── LLM service (only needed if USE_LLM = True) ───────────────────────────────
# Options:
#   'gemini'  → marker.services.gemini.GoogleGeminiService    (needs GOOGLE_API_KEY)
#   'claude'  → marker.services.claude.ClaudeService          (needs ANTHROPIC_API_KEY)
#   'openai'  → marker.services.openai.OpenAIService          (needs OPENAI_API_KEY)
#   'ollama'  → marker.services.ollama.OllamaService          (local, no key needed)
LLM_SERVICE = 'gemini'

# API keys — only fill in the one you need
GOOGLE_API_KEY    = ''   # for Gemini
ANTHROPIC_API_KEY = ''   # for Claude
OPENAI_API_KEY    = ''   # for OpenAI

# ── Validation ────────────────────────────────────────────────────────────────
assert os.path.exists(INPUT_FILE), f'❌ File not found: {INPUT_FILE}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Input        : {INPUT_FILE}')
print(f'✅ Output dir   : {OUTPUT_DIR}')
print(f'✅ Format       : {OUTPUT_FORMAT}')
print(f'✅ Pages        : {PAGE_RANGE or "all"}')
print(f'✅ LLM boost    : {USE_LLM}')
print(f'✅ Force OCR    : {FORCE_OCR}')

✅ Input        : /content/drive/MyDrive/tyt-biyoloji.pdf
✅ Output dir   : /content/drive/MyDrive/marker_output
✅ Format       : markdown
✅ Pages        : (9, 172)
✅ LLM boost    : False
✅ Force OCR    : False


## 5. 🚀 Run Marker
First run downloads ~2 GB of model weights. Subsequent runs use the cache.

In [ ]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser
import json, os, time, io
import numpy as np
from PIL import Image as PILImage

# ── Set API keys in environment ───────────────────────────────────────────────
if GOOGLE_API_KEY:
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
if ANTHROPIC_API_KEY:
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# ── Build config ──────────────────────────────────────────────────────────────
config = {
    'output_format': OUTPUT_FORMAT,
    'force_ocr'    : FORCE_OCR,
}

# PAGE_RANGE must be a string like '8-174' or None — never a tuple
# Marker is 0-indexed, so page 9 (1-indexed) = 8 (0-indexed)
if PAGE_RANGE is not None:
    if isinstance(PAGE_RANGE, tuple):
        # auto-convert (9, 175) → '8-174'
        config['page_range'] = f'{PAGE_RANGE[0] - 1}-{PAGE_RANGE[1] - 1}'
    else:
        config['page_range'] = str(PAGE_RANGE)

if USE_LLM:
    service_map = {
        'gemini' : 'marker.services.gemini.GoogleGeminiService',
        'claude' : 'marker.services.claude.ClaudeService',
        'openai' : 'marker.services.openai.OpenAIService',
        'ollama' : 'marker.services.ollama.OllamaService',
    }
    config['llm_service'] = service_map.get(LLM_SERVICE, LLM_SERVICE)

config_parser = ConfigParser(config)

# ── Load models (downloads on first run) ──────────────────────────────────────
print('[INFO] Loading models...')
t0 = time.time()
artifact_dict = create_model_dict()
print(f'[INFO] Models loaded in {time.time() - t0:.1f}s')

# ── Build converter ───────────────────────────────────────────────────────────
converter_kwargs = dict(
    config         = config_parser.generate_config_dict(),
    artifact_dict  = artifact_dict,
    processor_list = config_parser.get_processors(),
    renderer       = config_parser.get_renderer(),
)
if USE_LLM:
    converter_kwargs['llm_service'] = config_parser.get_llm_service()

converter = PdfConverter(**converter_kwargs)

# ── Run conversion ────────────────────────────────────────────────────────────
print(f'[INFO] Converting: {INPUT_FILE}')
t1 = time.time()
rendered = converter(INPUT_FILE)
elapsed  = time.time() - t1
print(f'[INFO] Conversion complete in {elapsed:.1f}s  ({elapsed/60:.1f} min)')

# ── Extract text and images ───────────────────────────────────────────────────
text, metadata, images = text_from_rendered(rendered)

# ── Save outputs ──────────────────────────────────────────────────────────────
base_name   = os.path.splitext(os.path.basename(INPUT_FILE))[0]
ext_map     = {'markdown': 'md', 'json': 'json', 'html': 'html', 'chunks': 'json'}
out_ext     = ext_map.get(OUTPUT_FORMAT, 'txt')
output_file = os.path.join(OUTPUT_DIR, f'{base_name}.{out_ext}')
meta_file   = os.path.join(OUTPUT_DIR, f'{base_name}_metadata.json')
images_dir  = os.path.join(OUTPUT_DIR, 'images')

# Main text output
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(text)
print(f'[INFO] Text saved  : {output_file}  ({len(text):,} chars)')

# Metadata
with open(meta_file, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f'[INFO] Meta saved  : {meta_file}')

# Images — save as PNG to avoid JPEG encoder argument overflow bug in Pillow
if images:
    os.makedirs(images_dir, exist_ok=True)
    saved, skipped = 0, 0
    for img_name, img in images.items():
        # strip original extension and force .png
        png_name = os.path.splitext(img_name)[0] + '.png'
        img_path = os.path.join(images_dir, png_name)
        try:
            PILImage.fromarray(np.array(img)).save(img_path, format='PNG')
            saved += 1
        except Exception as e:
            print(f'[WARN] Skipped {png_name}: {e}')
            skipped += 1
    print(f'[INFO] Images saved: {images_dir}  ({saved} saved, {skipped} skipped)')

print(f'\n✅ Done.')

[INFO] Loading models...
[INFO] Models loaded in 14.6s
[INFO] Converting: /content/drive/MyDrive/tyt-biyoloji.pdf


Recognizing Text: 100%|██████████| 139/139 [00:11<00:00, 11.77it/s]


[INFO] Conversion complete in 416.8s  (6.9 min)
[INFO] Text saved  : /content/drive/MyDrive/marker_output/tyt-biyoloji.md  (321,571 chars)
[INFO] Meta saved  : /content/drive/MyDrive/marker_output/tyt-biyoloji_metadata.json
[INFO] Images saved: /content/drive/MyDrive/marker_output/images  (559 saved, 0 skipped)

✅ Done.
